# Story 4 — The Sentiment Correlation
**Goal:** Prove (or disprove) that late deliveries cause bad customer reviews.

**Input:** `../data/02_delivered.parquet`  
**Output:** Charts saved to `../exports/`

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

COLOR_MAP  = {'On Time': '#2ecc71', 'Late': '#f39c12', 'Super Late': '#e74c3c'}
ORDER_CATS = ['On Time', 'Late', 'Super Late']
print('Libraries loaded.')

## Step 1 — Load data

In [ ]:
delivered = pd.read_parquet('../data/02_delivered.parquet')
print(f'Loaded: {len(delivered):,} delivered orders')
print(f'Orders with review scores: {delivered["review_score"].notna().sum():,}')

## Step 2 — Average review score by delivery status

In [ ]:
sentiment = (
    delivered.dropna(subset=['review_score'])
    .groupby('delivery_status')['review_score']
    .agg(['mean', 'median', 'count'])
    .reindex(ORDER_CATS)
    .reset_index()
)
sentiment.columns = ['delivery_status', 'avg_score', 'median_score', 'n_orders']

overall_avg = delivered['review_score'].mean()
print(f'Overall average review score: {overall_avg:.2f}/5\n')
print(sentiment.to_string(index=False))

## Step 3 — Bar chart + Box plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar(
    sentiment['delivery_status'], sentiment['avg_score'],
    color=[COLOR_MAP[s] for s in sentiment['delivery_status']],
    edgecolor='white', linewidth=1.5
)
axes[0].set_ylim(0, 5.5)
axes[0].set_ylabel('Average Review Score (1–5)')
axes[0].set_title('Average Review Score by Delivery Status', fontsize=13, fontweight='bold')
axes[0].axhline(y=overall_avg, color='grey', linestyle='--', label=f'Overall avg ({overall_avg:.2f})')
axes[0].legend()
for bar, val in zip(bars, sentiment['avg_score']):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.05,
                 f'{val:.2f}', ha='center', fontweight='bold', fontsize=11)

# Box plot
plot_data = delivered.dropna(subset=['review_score'])
groups = [plot_data[plot_data['delivery_status'] == s]['review_score'].values for s in ORDER_CATS]
bp = axes[1].boxplot(groups, labels=ORDER_CATS, patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
for patch, s in zip(bp['boxes'], ORDER_CATS):
    patch.set_facecolor(COLOR_MAP[s])
    patch.set_alpha(0.7)
axes[1].set_ylabel('Review Score')
axes[1].set_title('Review Score Distribution by Delivery Status', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../exports/fig_sentiment_by_status.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — Stacked bar: score distribution per status
Shows the *full picture* — not just averages but how reviews distribute (1★ to 5★) per group.

In [ ]:
score_dist = (
    delivered.dropna(subset=['review_score'])
    .groupby(['delivery_status', 'review_score'])
    .size()
    .reset_index(name='count')
)
score_dist['review_score'] = score_dist['review_score'].astype(str)

fig = px.bar(
    score_dist,
    x='delivery_status', y='count', color='review_score',
    barmode='stack',
    color_discrete_sequence=px.colors.sequential.RdYlGn,
    title='Review Score Distribution by Delivery Status',
    labels={'count': 'Number of Orders', 'delivery_status': 'Status', 'review_score': 'Score'},
    category_orders={'delivery_status': ORDER_CATS},
)
fig.show()

## Step 5 — Delay vs review score (binned scatter with trendline)
Each bubble = a 2-day delay bin. Size = number of orders in that bin.

In [ ]:
binned = delivered.dropna(subset=['review_score']).copy()
binned['delay_bin'] = pd.cut(binned['delivery_delay_days'], bins=range(-30, 80, 2), labels=False)
bin_summary = (
    binned.dropna(subset=['delay_bin'])
    .groupby('delay_bin')
    .agg(avg_review=('review_score', 'mean'), count=('order_id', 'count'))
    .reset_index()
)
bin_summary['delay_midpoint'] = bin_summary['delay_bin'] * 2 - 30 + 1

fig = px.scatter(
    bin_summary, x='delay_midpoint', y='avg_review',
    size='count', color='avg_review',
    color_continuous_scale='RdYlGn', range_color=[1, 5],
    title='Average Review Score vs Delivery Delay (bubble size = order volume)',
    labels={'delay_midpoint': 'Delay days (negative = early)', 'avg_review': 'Avg Review Score'},
    trendline='lowess',
)
fig.add_vline(x=0, line_dash='dash', line_color='black', annotation_text='On time')
fig.update_layout(coloraxis_showscale=False)
fig.show()

## Step 6 — Key finding

In [ ]:
on_time_avg = sentiment.loc[sentiment['delivery_status'] == 'On Time', 'avg_score'].values[0]
super_late_avg = sentiment.loc[sentiment['delivery_status'] == 'Super Late', 'avg_score'].values[0]
drop = on_time_avg - super_late_avg

print('─' * 55)
print(f' On Time orders avg review score:      {on_time_avg:.2f} / 5')
print(f' Super Late orders avg review score:   {super_late_avg:.2f} / 5')
print(f' Score drop for Super Late orders:     -{drop:.2f} points')
print('─' * 55)
print('\n✅ Conclusion: Late deliveries strongly correlate with lower review scores.')
print('   The CEO\'s hypothesis is confirmed by the data.')